# Crest Chart
Attempt to display a chart that shows the crest at each gauge for a single flood

y = stage heigh

x = river mile or gauge order

In [92]:
import pandas as pd
import numpy as np
import glob
import os
from pathlib import Path
import requests
import json

response = requests.get("https://floodzilla.com/api/client/APIGetLocationInfo?regionId=1")
gauges = response.json()
gauges_df = pd.DataFrame(gauges)
carn = next(g for g in gauges if g['id'] == "USGS-22")
floods_df = pd.DataFrame(carn['floodEvents'])
gauges_df['join_id'] = gauges_df['id']
gauges_df.loc[gauges_df['usgsSiteId'] >0, 'join_id'] = gauges_df['usgsSiteId'].astype(str)
gauges_df['riverMile'] = gauges_df['id'].str.extract(r'([0-9]+)', expand=True).astype(int)

#drop forks
gauges_df.drop(gauges_df[gauges_df['id'].isin(['USGS-SF17','USGS-MF11','USGS-NF10'])].index, inplace=True)


gauges_df.head(50)

,id,locationName,shortName,latitude,longitude,isOffline,rank,yMin,yMax,dischargeMin,...,isCurrentlyOffline,floodEvents,dischargeStageOne,dischargeStageTwo,maxChangeThreshold,roadSaddleHeight,roadDisplayName,locationImages,join_id,riverMile
3,USGS-38,Snoqualmie River - Below the Falls,Below the Falls,47.545077,-121.842316,False,22.0,0.00,25.00,100.0,...,False,"[{'id': 38, 'eventName': '2026 - March', 'from...",17000.0,20000.0,NaN,NaN,NaN,NaN,12144500,38
4,SVPA-37,Snoqualmie Falls Golf Course,Sno Falls Golf,47.561390,-121.878524,True,25.0,85.15,98.60,NaN,...,True,"[{'id': 38, 'eventName': '2026 - March', 'from...",NaN,NaN,2.0,88.60,lower parking lot,[7/c00f8509-cb25-4952-a185-3246ea5d39e1.png],SVPA-37,37
5,SVPA-36,Fall City- Neal Rd. SE,Neal Rd. SE,47.585410,-121.903805,True,29.0,69.10,83.44,NaN,...,True,"[{'id': 38, 'eventName': '2026 - March', 'from...",NaN,NaN,2.0,74.12,Neal Rd. SE,[106/c5b61817-396d-431d-a792-87acebd35318.jpg],SVPA-36,36
6,SVPA-34,"The ""T"" at W Snoqualmie River Rd SE",The T,47.587822,-121.926108,True,30.0,74.31,86.61,NaN,...,True,"[{'id': 38, 'eventName': '2026 - March', 'from...",NaN,NaN,NaN,75.85,W Snoqualmie River Rd SE,[1/85606e5d-6a29-43f1-8f08-4d402743c806.jpeg],SVPA-34,34
7,SVPA-33,Snoqualmie River at SE 19th Way,SE 19th Way,47.592762,-121.921028,True,45.0,61.13,90.13,NaN,...,True,"[{'id': 38, 'eventName': '2026 - March', 'from...",NaN,NaN,NaN,81.95,SE 19th Way,[8/034a5a2b-0fa4-44ed-802e-e1111ae83c0b.jpeg],SVPA-33,33
8,SVPA-29,W Snoqualmie River Rd NE near Jubilee Farm,Jubilee,47.611599,-121.934404,True,51.0,63.92,80.42,NaN,...,True,"[{'id': 38, 'eventName': '2026 - March', 'from...",NaN,NaN,NaN,75.67,West Snoqualmie River Rd. NE,[104/57540392-4627-4659-b2c6-5b7062f18834.jpg],SVPA-29,29
9,SVPA-26,W Snoqualmie River Rd NE at Blue Heron Golf Co...,Blue Heron Golf,47.625158,-121.933527,True,55.0,64.73,75.74,NaN,...,True,"[{'id': 38, 'eventName': '2026 - March', 'from...",NaN,NaN,NaN,66.51,W Snoqualmie River Rd NE,[29/b811050f-fe31-4435-b477-d34e1bac85cd.jpeg],SVPA-26,26
10,SVPA-25,Snoqualmie River at NE Tolt Hill Rd,Tolt Hill,47.638133,-121.928484,False,62.0,57.03,80.53,NaN,...,False,"[{'id': 38, 'eventName': '2026 - March', 'from...",NaN,NaN,NaN,68.43,Tolt Hill Rd.,NaN,SVPA-25,25
11,USGS-22,Snoqualmie River at Carnation,Carnation,47.665934,-121.925397,False,70.0,44.00,62.00,100.0,...,False,"[{'id': 38, 'eventName': '2026 - March', 'from...",16500.0,19400.0,NaN,NaN,NaN,NaN,12149000,22
12,SVPA-19,Stuart Landing - South of Oxbow Farm,Stuart Landing,47.688900,-121.963700,True,76.0,42.87,51.33,NaN,...,True,"[{'id': 38, 'eventName': '2026 - March', 'from...",NaN,NaN,NaN,43.94,lower farming access road,[111/c2a7f3ad-0e40-45b2-bafb-51f528856320.jpg],SVPA-19,19


In [93]:
files = list(Path("../crests").rglob("*.csv"))
raw_crests_df = pd.concat((pd.read_csv(f, sep="\t") for f in files))
raw_crests_df['location']= raw_crests_df['location'].astype(str)
raw_crests_df['utc']=pd.to_datetime(raw_crests_df['utc'])
raw_crests_df.head()

,Unnamed: 0,up_utc,down_utc,crest_hours,up_discharge,down_discharge,utc,stage,stage_qualifiers,location,discharge,discharge_qualifiers,id
0,0.0,2007-10-03 03:15:00,2007-10-03 06:45:00,3.50,10039.0,9660.0,NaT,NaN,NaN,NaN,NaN,NaN,NaN
1,1.0,2007-10-08 08:15:00,2007-10-08 11:00:00,2.75,6171.0,6720.0,NaT,NaN,NaN,NaN,NaN,NaN,NaN
2,2.0,2007-10-21 17:45:00,2007-10-21 20:45:00,3.00,4553.0,5010.0,NaT,NaN,NaN,NaN,NaN,NaN,NaN
3,3.0,2007-11-10 19:00:00,2007-11-11 00:00:00,5.00,2350.0,2540.0,NaT,NaN,NaN,NaN,NaN,NaN,NaN
4,4.0,2007-11-13 04:45:00,2007-11-13 08:45:00,4.00,3616.0,3740.0,NaT,NaN,NaN,NaN,NaN,NaN,NaN


In [94]:
crest_dfs = []
for flood in carn['floodEvents']:
    flood_df = raw_crests_df[(raw_crests_df['utc'] - pd.Timedelta(8, unit='h') >= flood['fromDate']) & (raw_crests_df['utc'] -  pd.Timedelta(8, unit='h') <= flood['toDate'])].copy()
    flood_df['flood'] = flood['eventName']
    crest_dfs.append(flood_df)
crest_df = pd.concat(crest_dfs)
crest_df = pd.merge(crest_df, gauges_df, left_on='location', right_on="join_id")    
crest_df = crest_df[~pd.isnull(crest_df['riverMile'])]
crest_df.sort_values(by=['flood','rank'], ascending=False, inplace = True)

#fix falls
crest_df['adjStage'] = crest_df['stage']
crest_df.loc[crest_df['location'] == '12144500', 'adjStage'] = crest_df['stage'] + 100

# filter dates
crest_df = crest_df[crest_df['utc'] > '2018-09-09']

crest_df['name'] = crest_df['shortName']
crest_df['gaugeId'] = crest_df['id_y']
# remove multi-crests
group_df = crest_df.groupby(['flood','name']).max()
group_df.reset_index(inplace=True)
group_df.sort_values(by=['flood','rank'], ascending=False, inplace = True)
group_df.head()

,flood,name,Unnamed: 0,up_utc,down_utc,crest_hours,up_discharge,down_discharge,utc,stage,...,dischargeStageOne,dischargeStageTwo,maxChangeThreshold,roadSaddleHeight,roadDisplayName,locationImages,join_id,riverMile,adjStage,gaugeId
179,2026 - March,Duvall,NaN,NaN,NaN,NaN,NaN,NaN,2026-03-25 17:00:00,41.67,...,NaN,NaN,NaN,NaN,NaN,NaN,12150400,9,41.67,USGS-9
184,2026 - March,NE 138th St,NaN,NaN,NaN,NaN,NaN,NaN,2026-03-21 12:29:57,46.81,...,NaN,NaN,NaN,36.22,NE 138th St,[13/e52d45e2-358b-4c5b-82b5-4c3f9c801147.png],SVPA-12,12,46.81,SVPA-12
183,2026 - March,NE 124th St (W),NaN,NaN,NaN,NaN,NaN,NaN,2026-03-21 13:00:00,49.65,...,NaN,NaN,NaN,45.25,NE 124th St,NaN,SVPA-15B,15,49.65,SVPA-15B
182,2026 - March,NE 124th St (E),NaN,NaN,NaN,NaN,NaN,NaN,2026-03-21 10:00:10,49.62,...,NaN,NaN,NaN,43.25,NE 124th St East,[21/01cae7f2-0957-4008-9928-e698ce4a1761.jpeg],SVPA-15,15,49.62,SVPA-15
181,2026 - March,NE 100th St,NaN,NaN,NaN,NaN,NaN,NaN,2026-03-21 10:32:08,53.55,...,NaN,NaN,NaN,46.40,NE 100th St.,[12/d377e57e-4619-45f7-b025-536e9d7b2e00.jpg],SVPA-17,17,53.55,SVPA-17


In [95]:
# overrides
# null  -> suppress a false-positive crest
# {"stage": X} -> correct the crest height (e.g. sensor datum shift)
with open('../data_files//historic-crest-overrides.json') as f:
   overrides = json.load(f)
for gauge_id in overrides:
   for flood, val in overrides[gauge_id]['overrides'].items():
      mask = (group_df['gaugeId'] == gauge_id) & (group_df['flood'] == flood)
      if val is None:
         group_df.drop(group_df[mask].index, inplace=True)
      elif 'stage' in val:
         group_df.loc[mask, 'stage'] = val['stage']
         group_df.loc[mask, 'adjStage'] = val['stage'] + np.where(group_df.loc[mask, 'location'] == '12144500', 100, 0)

In [96]:
import plotly.express as px

fig = px.line(group_df, x="riverMile", y="adjStage", title='Recent Floods', color="flood", markers=True, height=1000)
fig['layout']['xaxis']['autorange'] = "reversed"

fig.show()


In [97]:
group_df.sort_values(by=['flood','rank'], ascending=False, inplace = True)
#min
tmp_group = group_df.groupby(['name']).min()
tmp_group['minStage'] = tmp_group['stage']
group_df = pd.merge(group_df, tmp_group[['location','minStage']], on="location")
#max
tmp_group = group_df.groupby(['name']).max()
tmp_group['maxStage'] = tmp_group['stage']

group_df = pd.merge(group_df, tmp_group[['location','maxStage']], on="location")
group_df['normStage'] = group_df['stage'] - group_df['minStage']
group_df['stageRangePrecent'] = 100 * (group_df['stage'] - group_df['minStage']) / (group_df['maxStage'] - group_df['minStage'])

group_df.head()

,flood,name,Unnamed: 0,up_utc,down_utc,crest_hours,up_discharge,down_discharge,utc,stage,...,roadDisplayName,locationImages,join_id,riverMile,adjStage,gaugeId,minStage,maxStage,normStage,stageRangePrecent
0,2026 - March,Duvall,NaN,NaN,NaN,NaN,NaN,NaN,2026-03-25 17:00:00,41.67,...,NaN,NaN,12150400,9,41.67,USGS-9,37.23,45.23,4.44,55.500000
1,2026 - March,NE 138th St,NaN,NaN,NaN,NaN,NaN,NaN,2026-03-21 12:29:57,46.81,...,NE 138th St,[13/e52d45e2-358b-4c5b-82b5-4c3f9c801147.png],SVPA-12,12,46.81,SVPA-12,41.07,49.25,5.74,70.171149
2,2026 - March,NE 124th St (W),NaN,NaN,NaN,NaN,NaN,NaN,2026-03-21 13:00:00,49.65,...,NE 124th St,NaN,SVPA-15B,15,49.65,SVPA-15B,44.47,52.47,5.18,64.750000
3,2026 - March,NE 100th St,NaN,NaN,NaN,NaN,NaN,NaN,2026-03-21 10:32:08,53.55,...,NE 100th St.,[12/d377e57e-4619-45f7-b025-536e9d7b2e00.jpg],SVPA-17,17,53.55,SVPA-17,48.20,55.73,5.35,71.049137
4,2026 - March,Stuart Landing,NaN,NaN,NaN,NaN,NaN,NaN,2026-03-21 10:32:06,49.88,...,lower farming access road,[111/c2a7f3ad-0e40-45b2-bafb-51f528856320.jpg],SVPA-19,19,49.88,SVPA-19,49.88,49.88,0.00,NaN


In [98]:
import plotly.express as px

# order the categorical x-axis by rank (ascending rank == descending river mile,
# matching the riverMile chart: upstream on the left, Duvall on the right)
name_order = group_df.sort_values('rank')['name'].drop_duplicates().tolist()

fig = px.line(group_df, x="name", y="normStage", title='Floods', color="flood", markers=True, height=1000, text="stage",
              category_orders={"name": name_order})

fig.show()

In [99]:
import plotly.express as px

# order the categorical x-axis by rank (ascending rank == descending river mile,
# matching the riverMile chart: upstream on the left, Duvall on the right)
name_order = group_df.sort_values('rank')['name'].drop_duplicates().tolist()

fig = px.line(group_df, x="name", y="stageRangePrecent", title='Floods', color="flood", markers=True, height=1000, text="stage",
              category_orders={"name": name_order})

fig.show()

In [100]:
for_export_df =  group_df[['gaugeId', 'name', 'shortName', 'flood', 'utc','stage','discharge','rank','yMin','yellowStage','riverMile','adjStage','minStage','maxStage','normStage','stageRangePrecent']]
for_export_df.to_csv('../data_files/historic_crests_cleaned.csv')
